<a href="https://colab.research.google.com/github/jakemaier1/GA_Capstone_2026/blob/main/GA_Taxiway_robot_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Install Ultralytics dependencies securely
!pip install -q ultralytics onnx onnxslim onnxruntime
print("✅ Environment dependencies configured successfully!")

✅ Environment dependencies configured successfully!


In [14]:
import zipfile
import os
import yaml

# Extract runway dataset
zip_file_name = 'Airport Sign Labels.v2-five-signs-180-images-yolov8.yolov8.zip'
extract_path = '/content/runway_dataset'

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset successfully extracted!")

# Configure paths inside data.yaml
yaml_path = '/content/runway_dataset/data.yaml'
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)
    config['path'] = extract_path
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    print("✅ data.yaml configured successfully!")

✅ Dataset successfully extracted!
✅ data.yaml configured successfully!


In [15]:
from ultralytics import YOLO

# Initialize pre-trained YOLO11 Nano model
model_yolo11 = YOLO('yolo11n.pt')

# Train the model
results_yolo11 = model_yolo11.train(
    data='/content/runway_dataset/data.yaml',
    epochs=30,
    imgsz=416,
    batch=8,
    name='runway_signs_yolo11_nano',
    plots=True
)

Ultralytics 8.4.209 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=runway_signs_yolo11_nano-2, nbs=64, nms=None, opset=None

In [16]:
# Evaluate performance of YOLO11 nano
metrics_yolo11 = model_yolo11.val()

print("\n" + "="*40)
print(f"YOLO11 Nano mAP50: {metrics_yolo11.box.map50:.4f}")
print(f"YOLO11 Nano mAP50-95: {metrics_yolo11.box.map:.4f}")
print("="*40)

Ultralytics 8.4.209 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 922.1±232.0 MB/s, size: 26.2 KB)
val: Scanning /content/runway_dataset/valid/labels.cache... 36 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 8.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7it/s 1.7s
                   all         36         36      0.972       0.94       0.99      0.907
Runway_36_Direction_Right          8          8      0.912          1      0.995      0.901
         Runway_36_Red          6          6      0.966          1      0.995      0.895
             Taxiway_B         11         11          1      0.953      0.995      0.951
Taxiway_B_Direction_Left          6          6          1      0.749      0.972      0.907
Terminal_A_Direction_Right          5   

### **3. Export Weights to ONNX Format & Download**
This block will convert your optimal PyTorch checkpoint into a serialized ONNX graph and trigger an automatic browser download.

In [17]:
from ultralytics import YOLO
import os
from google.colab import files

best_30epoch_path = 'runs/detect/runway_signs_yolo11_nano/weights/best.pt'

if os.path.exists(best_30epoch_path):
    # Load best trained weights
    model = YOLO(best_30epoch_path)

    # Export model to ONNX
    print("📦 Exporting best 30-epoch PyTorch model to ONNX...")
    onnx_output_path = model.export(format='onnx', imgsz=416)

    if os.path.exists(onnx_output_path):
        print(f"✅ Export complete: {onnx_output_path}")
        files.download(onnx_output_path)
        print("📥 Downloading your best.onnx file directly to your machine!")
    else:
        print("❌ Export task did not generate the expected ONNX file.")
else:
    print(f"❌ Could not find the best weights at: {best_30epoch_path}")

📦 Exporting best 30-epoch PyTorch model to ONNX...
Ultralytics 8.4.209 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 2.7 GFLOPs

PyTorch: starting from 'runs/detect/runway_signs_yolo11_nano/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 9, 3549) (25.3 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 3.6s, saved as 'runs/detect/runway_signs_yolo11_nano/weights/best.onnx' (10.1 MB)

Export complete (4.0s)
Results saved to /content/runs/detect/runway_signs_yolo11_nano/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/runway_signs_yolo11_nano/weights/best.onnx imgsz=416 
Validate:        yolo val task=detect model=runs/detect/runway_signs_yolo11_nano/weights/best.onnx imgsz=416 data=/content/runway_dataset/data.yaml 
Visualize:       https://netron.app
✅ Export complete:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading your best.onnx file directly to your machine!


---
*This notebook has been cleaned of unused legacy test pipelines as requested.*

In [18]:
# Unused training cells cleared.

In [19]:
# Unused validation cells cleared.